# 第二次作业 02 旧来源复核与直达采集

保存旧来源复核和第一版直达采集流程；这里的历史报错输出原样保留。

原编号与新文件名的对应关系、外部依赖和执行说明见 [第二次作业导航](hw02_README.md)。

先运行本文件的**环境初始化**单元格；重启内核后也应先运行初始化。各节按原顺序排列，依赖上一步生成的变量时会在节前列出。

已有输出和执行编号是整理前的运行记录，未重新生成；新增初始化单元格没有历史输出。

**历史采集流程**：2026年11月的原采集窗口已结束。请结合已保存结果阅读，勿将“全部运行”视为重新采集入口；代码中的日期、截止条件和来源区分原样保留。

[上一份 01](hw02_01_environment_check.ipynb) · [下一份 03](hw02_03_collection_resume.ipynb)

## 环境初始化

此单元格集中加载公共依赖并检查脚本位置；不执行采集、安装或模型拟合。

In [ ]:
# 标准库
import importlib
import sys
from pathlib import Path

# 第三方库
import pandas as pd
from IPython.display import display

# 仅在这里配置本机项目位置。脚本和来源数据仍保留在原 railway_analysis 项目中。
ROOT = Path(r"D:\yangz\大三上\机器学习\第二次作业\railway_analysis")
EXPECTED_PYTHON = Path(r"D:\anaconda\envs\pda312\python.exe")
SCRIPTS_DIR = ROOT / "scripts"

if Path(sys.executable).resolve() != EXPECTED_PYTHON.resolve():
    raise RuntimeError("请选择 pda312 内核，再从本文件的环境初始化单元格开始运行。")
if not SCRIPTS_DIR.is_dir():
    raise FileNotFoundError(f"未找到作业脚本目录：{SCRIPTS_DIR}。请核对上方 ROOT 配置。")

REQUIRED_LOCAL_MODULES = (
    "stage1_collect",
    "stage1b_transfer",
    "stage1c_collect",
    "stage1c_review",
)
missing_modules = [
    name for name in REQUIRED_LOCAL_MODULES
    if not (SCRIPTS_DIR / f"{name}.py").is_file()
]
if missing_modules:
    raise FileNotFoundError(f"作业脚本缺失：{', '.join(missing_modules)}")

# 重复运行初始化时不重复插入同一路径；业务模块仍在对应步骤导入和 reload。
scripts_path = str(SCRIPTS_DIR)
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)


## 问题1—辅助代码2：环境、路径及模块加载（无需单独截图）

In [2]:
# 输入：已存在的Python内核和作业scripts目录。
# 输出：解释器、日期配置和995组采集计划数量。
# 顺序：本段必须先运行；若更换内核或重启内核，重新运行本段。

import stage1_collect
import stage1b_transfer
import stage1c_review
import stage1c_collect
for module in (stage1_collect, stage1b_transfer, stage1c_review, stage1c_collect):
    importlib.reload(module)
print("当前解释器：", sys.executable)
print("旧批次日期：2026-11-10；本批新日期：2026-11-18")
print("新采集计划：992个核心城市有向对＋3组辅助接入，共995组。")


当前解释器： D:\anaconda\envs\pda312\python.exe
旧批次日期：2026-11-10；本批新日期：2026-11-18
新采集计划：992个核心城市有向对＋3组辅助接入，共995组。


## 问题1—核心代码7：旧数据复核与未知时刻下界【必须运行并截图】

前置变量或模块：`stage1c_review`。

In [4]:
# 功能：重建原始响应并与CSV核对，检查2条未知时刻能否改善已有最小候选。
review = stage1c_review.review(actor="user")
REVIEW_OUT = Path(review["output_dir"])
bounded = pd.read_csv(REVIEW_OUT / "bounded_transfer_scenarios.csv", encoding="utf-8-sig")
display(bounded[["threshold_min", "known_candidate_count", "known_subset_min_min",
                 "minimum_missing_lower_bound_min", "named_missing_rows_cannot_improve"]])
display(pd.read_csv(REVIEW_OUT / "missing_time_lower_bounds.csv", encoding="utf-8-sig")
        [["train_code", "query_date", "latest_first_departure", "lower_bound_min"]])
display(pd.read_csv(REVIEW_OUT / "access_endpoint_quality.csv", encoding="utf-8-sig"))
print("已核验有效记录：", review["validated_records"], "；未知时刻记录：", review["missing_records"])
print("保存位置：", REVIEW_OUT)
print("两条时刻仍未知；结果限于采集范围和换乘假设，不等于全国最短或实际换乘保证。")


,threshold_min,known_candidate_count,known_subset_min_min,minimum_missing_lower_bound_min,named_missing_rows_cannot_improve
0,30,102,1971,4572.0,True
1,60,99,2075,4572.0,True
2,90,98,2075,4572.0,True


,train_code,query_date,latest_first_departure,lower_bound_min
0,Z8981,2026-11-14,2026-11-10T19:48:00+08:00,4572.0
1,Y839,2026-11-14,2026-11-10T19:48:00+08:00,4572.0


,query_id,returned_records,exact_requested_endpoints,other_endpoints
0,A01,135,81,54
1,A02,101,40,61
2,A03,99,39,60


已核验有效记录： 357 ；未知时刻记录： 2
保存位置： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_offline_review_20261113_123835_808647
两条时刻仍未知；结果限于采集范围和换乘假设，不等于全国最短或实际换乘保证。


## 问题1—核心代码8：完整城市对直达采集与断点保存【必须运行并截图】

前置变量或模块：`stage1c_collect`。

此节保留历史失败输出，用于说明当时的采集/核验状态；不表示本次整理已重新执行或修复外部来源。

In [5]:
# 功能：在新日期查询995组方向，逐请求保存原始响应、解析结果和检查点。
MAX_QUERIES = 995  # 可改为100先采一批；之后再次运行本段续采
jobs = sorted((ROOT / "data/raw").glob("user_stage1c_*"))
resume_job = str(jobs[-1]) if jobs else None
collection = stage1c_collect.collect(actor="user", job_dir=resume_job, max_queries=MAX_QUERIES)
COLLECTION_OUT = Path(collection["output_dir"])
for key in ["query_date", "completed_queries", "remaining_queries", "train_records",
            "unavailable_timetable_records", "outside_candidate_record_count", "blocking_error"]:
    print(f"{key}: {collection[key]}")
print("保存位置：", COLLECTION_OUT)
print("matrix_ready:", collection["matrix_ready"], "（直达查询表尚非最短行程矩阵）")


采集目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\raw\user_stage1c_20261113_123705_523939_reviewed
196/995 长春→哈尔滨: validated_records; valid=111
197/995 长春→上海: validated_records; valid=8
198/995 长春→南京: validated_records; valid=10
199/995 长春→杭州: validated_records; valid=5
220/995 长春→香港: no_direct_train_returned_not_unreachable; valid=0
245/995 哈尔滨→拉萨: no_direct_train_returned_not_unreachable; valid=0
270/995 上海→海口: validated_records; valid=1
295/995 南京→济南: validated_records; valid=109
320/995 杭州→哈尔滨: validated_records; valid=5
345/995 合肥→天津: validated_records; valid=14
370/995 合肥→兰州: validated_records; valid=13
395/995 福州→重庆: validated_records; valid=6
420/995 南昌→郑州: validated_records; valid=18
445/995 济南→上海: validated_records; valid=75
470/995 郑州→石家庄: validated_records; valid=158
495/995 郑州→西宁: validated_records; valid=16
520/995 武汉→成都: validated_records; valid=33
545/995 长沙→郑州: validated_records; valid=93
570/995 广州→南京: validated_records; valid=6
595/995 南宁→呼和浩特: validated_records; v

ValueError: Parsed checkpoint differs from raw response: N07_N06